# Train on a Colab GPU

Runs `scripts/train.py` on a Colab GPU runtime instead of the local CPU. It clones the repo,
downloads the PhiUSIIL dataset (UCI) and the Tranco list, builds the splits, trains and evaluates.

Connect this notebook to a Colab **GPU** runtime first, either in the VS Code Colab extension or
in the Colab web UI under *Runtime → Change runtime type*.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a GPU type"
print(torch.cuda.get_device_name(0))

NVIDIA A100-SXM4-40GB


In [ ]:
REPO = "https://github.com/Kluey/URL-Phishing-Classifier.git"
BRANCH = "hostname-input"

%cd /content
!rm -rf URL-Phishing-Classifier && git clone -q --depth 1 -b {BRANCH} {REPO}
%cd /content/URL-Phishing-Classifier
!git log --oneline -1
!pip install -q -e ".[pretrain]"

/content
/content/URL-Phishing-Classifier
82b9f5c (grafted, HEAD -> hostname-input, origin/hostname-input) chore(notebook): compare configurations by validation accuracy
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for phishing-classifier (pyproject.toml) ... done


In [ ]:
!mkdir -p data model
!wget -q -O /tmp/phiusiil.zip "https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip"
!unzip -o -q /tmp/phiusiil.zip -d data && ls -la data

total 55532
drwxr-xr-x  2 root root     4096 Nov  6 04:37 .
drwxr-xr-x 11 root root     4096 Nov  6 04:37 ..
-rw-------  1 root root 56854345 Mar  4  2024 PhiUSIIL_Phishing_URL_Dataset.csv


Builds the hostname splits (PhiUSIIL + Tranco subdomains, all balanced data) and trains with the
defaults: the last transformer block, the final norm and the head, for 2 epochs. Fine-tuning 2, 4
or 6 blocks for 3 epochs raised validation accuracy slightly (up to 79.3% vs 77.8%) but did worse
on the test split and the real-world check, so 1 block is kept.

In [ ]:
!python scripts/train.py --prepare-data --pretrained-dir /content/gpt2 2>&1 | tee model/train.log

In [ ]:
!python scripts/evaluate.py --split test
!python scripts/evaluate.py --split real_world

Device: cuda

Test metrics (34329 URLs, phishing = positive class, threshold 0.5):
  accuracy                0.7864
  precision_phishing      0.8978
  recall_phishing         0.6726
  false_positive_rate     0.0859
  mcc                     0.5988
  brier                   0.1472
  roc_auc                 0.8795
  pr_auc                  0.9087
  recall_at_1.0%_fpr      0.4729
  recall_at_0.1%_fpr      0.3038

Saved metrics to /content/URL-Phishing-Classifier/model/phishing_classifier.test_metrics.json
Device: cuda

Real_world metrics (70 URLs, phishing = positive class, threshold 0.5):
  accuracy                0.9286
  precision_phishing      0.8000
  recall_phishing         1.0000
  false_positive_rate     0.1000
  mcc                     0.8485
  brier                   0.0529
  roc_auc                 1.0000
  pr_auc                  1.0000
  recall_at_1.0%_fpr      1.0000
  recall_at_0.1%_fpr      1.0000

Misclassified (5):
  0.511  label=1  https://github.com/pytorch/pytorch/iss

Files to bring back, each into the same folder in the local project. In VS Code, use the Colab
sidebar → **Contents** → right-click a file → **Download...**.

In [ ]:
!cd /content/URL-Phishing-Classifier && ls -la model/phishing_classifier.pth model/*.json model/train.log data/*.csv data/splits.json | grep -v -e PhiUSIIL -e tranco

-rw-r--r-- 1 root root        88 Nov  6 04:39 data/splits.json
-rw-r--r-- 1 root root   1229948 Nov  6 04:39 data/test.csv
-rw-r--r-- 1 root root   4196854 Nov  6 04:39 data/train.csv
-rw-r--r-- 1 root root    443173 Nov  6 04:39 data/validation.csv
-rw-r--r-- 1 root root 497821235 Nov  6 05:00 model/phishing_classifier.pth
-rw-r--r-- 1 root root       460 Nov  6 05:00 model/phishing_classifier.real_world_metrics.json
-rw-r--r-- 1 root root       561 Nov  6 05:00 model/phishing_classifier.test_metrics.json
-rw-r--r-- 1 root root       600 Nov  6 05:00 model/phishing_classifier.train.json
-rw-r--r-- 1 root root    105203 Nov  6 05:00 model/train.log
